# Drifting ICNN

In [1]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import random
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from typing import Tuple, List

In [2]:
def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)

### Hyperparameters

In [ ]:
METHODS = ["kernel", "ot_direct", "icnn"]
NUM_ITERS = 3000
BATCH_SIZE = 128
LATENT_DIM = 64
NOISE_DIM = 64
LR_GEN = 2e-4
LR_VAE = 1e-3
INNER_STEPS = 5
VAE_WARMUP_STEPS = 400
SEED = 42
seed_everything(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MNIST_ROOT = "./data"
SAVE_PATH = "results/mnist/mnist_drifting_results.png"

# ICNN architecture requested for the MNIST scaling experiment.
# These are hidden layer widths, not the latent dimensionality.
ICNN_HIDDEN_DIMS = [512, 256, 128, 64]

# Optional asymmetric generator widths. The output is still LATENT_DIM.
GEN_HIDDEN_DIMS = [512, 256, 128]

# Sinkhorn regularization is now dimensionless because we normalize the cost matrix.
# Tune this rather than relying on the common 0.05 default.
SINKHORN_REG = 0.2
SINKHORN_REG_GRID = [0.03, 0.05, 0.1, 0.2, 0.5, 1.0]
SINKHORN_NUM_ITERS = 80
NORMALIZE_OT_COST = True


### 1. The VAE (Encoder/Decoder)
We need this to map Images <-> Latent Space. 
The Drifting Model operates ONLY in the latent space.

In [4]:
class MNISTVAE(nn.Module):
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # 28 -> 14
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # 14 -> 7
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),# 7 -> 4
            nn.SiLU(),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.SiLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)

        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.SiLU(),
            nn.Linear(256, 128 * 4 * 4),
            nn.SiLU(),
        )
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (128, 4, 4)),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=0), # 4 -> 7
            nn.SiLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),                     # 7 -> 14
            nn.SiLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),                      # 14 -> 28
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor):
        h = self.decoder_fc(z)
        return self.decoder(h)

    def forward(self, x: torch.Tensor):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar



class LatentGenerator(nn.Module):
    """
    Asymmetric latent generator.

    With hidden_dims=(512, 256, 128), the Linear stack is:
        NOISE_DIM -> 512 -> 256 -> 128 -> LATENT_DIM
    For the current MNIST setup LATENT_DIM=64, so this gives the requested
    512 -> 256 -> 128 -> 64 progression at the generator output.
    """
    def __init__(self, noise_dim: int = 64, latent_dim: int = 64,
                 hidden_dims: Tuple[int, ...] = (512, 256, 128)):
        super().__init__()
        dims = [noise_dim, *hidden_dims, latent_dim]
        layers = []
        for in_dim, out_dim in zip(dims[:-2], dims[1:-1]):
            layers += [nn.Linear(in_dim, out_dim), nn.SiLU()]
        layers.append(nn.Linear(dims[-2], dims[-1]))
        self.net = nn.Sequential(*layers)
        self._init_output_scale(target_std=1.0)

    def _init_output_scale(self, target_std: float):
        with torch.no_grad():
            device = self.net[0].weight.device
            test_input = torch.randn(2000, self.net[0].in_features, device=device)
            test_output = self.net(test_input)
            current_std = test_output.std().item()
            if current_std > 1e-6:
                scale_factor = target_std / current_std
                self.net[-1].weight.mul_(scale_factor)
                self.net[-1].bias.mul_(scale_factor)

    def forward(self, eps: torch.Tensor):
        return self.net(eps)


### 2. ICNN & Sinkhorn (Same logic, adapted for variable dimensions)

In [5]:
def normalized_cost(x: torch.Tensor, y: torch.Tensor,
                    p: int = 2, squared: bool = True,
                    normalize: bool = True):
    """
    Pairwise cost with optional scale normalization.

    In latent spaces, raw ||x-y||² grows with dimension and latent scale.
    Normalizing by the batch median makes Sinkhorn `reg` dimensionless and
    avoids the high-dimensional underflow caused by exp(-C / reg).
    """
    C = torch.cdist(x, y, p=p)
    if squared:
        C = C ** 2
    if normalize:
        scale = C.detach().median().clamp_min(1e-6)
        C = C / scale
    return C


def sinkhorn(C: torch.Tensor, reg: float = 0.2, num_iters: int = 80):
    """
    Stable log-domain Sinkhorn plan for balanced minibatches.

    Returns a plan with approximately uniform marginals. The later row
    normalization used for barycentric targets is still kept because it is
    numerically convenient and works for unequal batch sizes too.
    """
    if reg <= 0:
        raise ValueError("Sinkhorn regularization `reg` must be positive.")

    n, m = C.shape
    log_K = -C / reg
    log_u = torch.zeros(n, device=C.device, dtype=C.dtype)
    log_v = torch.zeros(m, device=C.device, dtype=C.dtype)
    log_a = torch.full((n,), -math.log(n), device=C.device, dtype=C.dtype)
    log_b = torch.full((m,), -math.log(m), device=C.device, dtype=C.dtype)

    for _ in range(num_iters):
        log_u = log_a - torch.logsumexp(log_K + log_v.unsqueeze(0), dim=1)
        log_v = log_b - torch.logsumexp(log_K.transpose(0, 1) + log_u.unsqueeze(0), dim=1)

    log_P = log_u.unsqueeze(1) + log_K + log_v.unsqueeze(0)
    return torch.exp(log_P)


def sinkhorn_barycentric_target(x: torch.Tensor, y: torch.Tensor,
                                reg: float = SINKHORN_REG,
                                num_iters: int = SINKHORN_NUM_ITERS,
                                normalize_cost: bool = NORMALIZE_OT_COST):
    """
    Compute y_bar_i = sum_j P_ij y_j / sum_j P_ij.
    This is shared by `ot_direct` and `icnn` so both use the same regularization.
    """
    C = normalized_cost(x, y, squared=True, normalize=normalize_cost)
    P = sinkhorn(C, reg=reg, num_iters=num_iters)
    P_row = P / (P.sum(1, keepdim=True).clamp_min(1e-8))
    y_target = P_row @ y
    return y_target, C, P


In [6]:
class ICNN(nn.Module):
    """
    ICNN: ψ(x) = (μ/2)||x||² + skip(x) + hidden(h(x))
    so T(x) = ∇ψ = μx + ∇(skip) + ∇(hidden).
    Init: μ=1, skip=0, W_z≈0 → T(x) ≈ x (near-identity).
    """
    def __init__(self, input_dim: int, hidden_dims: List[int] = [512, 256, 128, 64],
                 beta: float = 20.0, strong_convexity: float = 1.0):
        super().__init__()
        self.input_dim = input_dim
        self.beta = beta
        self.num_layers = len(hidden_dims)

        # Keep μ positive by parameterizing it through softplus.
        # This is safer than optimizing a clamped raw parameter.
        init_raw_mu = math.log(math.exp(strong_convexity) - 1.0) if strong_convexity > 0 else -10.0
        self.raw_strong_convexity = nn.Parameter(torch.tensor(init_raw_mu, dtype=torch.float32))

        self.input_skip = nn.Linear(input_dim, 1, bias=True)
        self.Wz_0 = nn.Linear(input_dim, hidden_dims[0], bias=True)
        self.Wy = nn.ModuleList()
        self.Wz = nn.ModuleList()
        for i in range(1, self.num_layers):
            self.Wy.append(nn.Linear(hidden_dims[i-1], hidden_dims[i], bias=False))
            self.Wz.append(nn.Linear(input_dim, hidden_dims[i], bias=True))
        self.Wy_out = nn.Linear(hidden_dims[-1], 1, bias=False)
        self.hidden_bias = nn.Parameter(torch.zeros(1))
        self._init_identity()

    @property
    def strong_convexity(self):
        return F.softplus(self.raw_strong_convexity)

    def _init_identity(self):
        """Initialize parameters so T(x)=∇ψ(x) starts close to identity."""
        nn.init.zeros_(self.input_skip.weight)
        nn.init.zeros_(self.input_skip.bias)
        nn.init.zeros_(self.Wz_0.weight)
        nn.init.constant_(self.Wz_0.bias, -1.0)

        for layer in self.Wz:
            nn.init.zeros_(layer.weight)
            nn.init.constant_(layer.bias, -1.0)

        # Initialize hidden-to-hidden and hidden-to-output layers positive.
        for layer in list(self.Wy) + [self.Wy_out]:
            fan_in = layer.weight.shape[1]
            mu_w = fan_in ** (-0.5)
            sigma2_w = fan_in ** (-1.0)
            sigma2_ln = np.log(1.0 + sigma2_w / mu_w**2)
            mu_ln = np.log(mu_w) - 0.5 * sigma2_ln
            with torch.no_grad():
                layer.weight.copy_(torch.exp(
                    torch.tensor(mu_ln, device=layer.weight.device, dtype=layer.weight.dtype)
                    + np.sqrt(sigma2_ln) * torch.randn_like(layer.weight)
                ))
        nn.init.zeros_(self.hidden_bias)

    def softplus(self, x):
        return F.softplus(x, beta=self.beta)

    def forward(self, z):
        mu = self.strong_convexity.to(z.device, z.dtype)
        quad = 0.5 * mu * (z**2).sum(-1, keepdim=True)
        skip = self.input_skip(z)
        h = self.softplus(self.Wz_0(z))
        for i in range(len(self.Wy)):
            h = self.softplus(self.Wy[i](h) + self.Wz[i](z))
        hidden = self.Wy_out(h) + self.hidden_bias
        return quad + skip + hidden

    @torch.enable_grad()
    def gradient(self, z, create_graph=None):
        """Compute T(z)=∇_z ψ(z)."""
        if create_graph is None:
            create_graph = self.training
        z_in = z.detach().clone().requires_grad_(True)
        psi = self.forward(z_in)
        return torch.autograd.grad(psi.sum(), z_in, create_graph=create_graph)[0]

    def project_weights(self):
        """Project W_y weights to be non-negative to maintain convexity."""
        with torch.no_grad():
            for layer in list(self.Wy) + [self.Wy_out]:
                layer.weight.clamp_(min=0.0)


class ICNNDriftField:
    """
    V(x) = ∇ψ_ω(x) - x.

    The ICNN is warm-started across outer iterations; only the Sinkhorn
    barycentric target changes batch by batch.
    """
    def __init__(self, dim, hidden_dims=[512, 256, 128, 64],
                 inner_steps=5, inner_lr=1e-3,
                 sinkhorn_reg=0.2, sinkhorn_iters=80,
                 normalize_cost=True, strong_convexity=1.0):
        self.icnn = ICNN(dim, hidden_dims, strong_convexity=strong_convexity)
        self.optimizer = optim.Adam(self.icnn.parameters(), lr=inner_lr)
        self.inner_steps = inner_steps
        self.sinkhorn_reg = sinkhorn_reg
        self.sinkhorn_iters = sinkhorn_iters
        self.normalize_cost = normalize_cost

    def to(self, device):
        self.icnn = self.icnn.to(device)
        self.optimizer = optim.Adam(self.icnn.parameters(), lr=self.optimizer.defaults['lr'])
        return self

    def compute_V(self, x_gen, y_pos):
        x = x_gen.detach()
        y = y_pos.detach()

        with torch.no_grad():
            y_target, _, _ = sinkhorn_barycentric_target(
                x, y,
                reg=self.sinkhorn_reg,
                num_iters=self.sinkhorn_iters,
                normalize_cost=self.normalize_cost,
            )

        self.icnn.train()
        for _ in range(self.inner_steps):
            self.optimizer.zero_grad(set_to_none=True)
            T_x = self.icnn.gradient(x, create_graph=True)
            loss = ((T_x - y_target) ** 2).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.icnn.parameters(), 5.0)
            self.optimizer.step()
            self.icnn.project_weights()

        self.icnn.eval()
        T_x = self.icnn.gradient(x, create_graph=False)
        return (T_x.detach() - x).detach()


### 3. Baselines (Kernel & OT Direct) - Adapted for Latent Space

In [7]:
def compute_V_kernel(x_gen: torch.Tensor, y_pos: torch.Tensor,
                     tau_list: Tuple[float, ...] = (0.02, 0.05, 0.2)):
    """
    Compute the drifting field V(x) following the original Drifting Model (Deng et al.)
    Algorithm 2 / Eq. 11 of the paper.

    Returns V (detached), such that the training loss is:
        target = sg(x_gen + V)
        loss = ||x_gen - target||²

    The loss value equals ||V||², which DECREASES as q → p.
    """
    N, D = x_gen.shape
    M = y_pos.shape[0]
    x = x_gen.detach()
    y = y_pos.detach()

    # Targets: [negatives (=generated), positives (=data)]
    targets = torch.cat([x, y], dim=0)  # [N+M, D]

    # ── Pairwise ℓ₂ distances between generated samples and all targets ──
    dist = torch.cdist(x, targets)  # [N, N+M]

    # ── Distance normalization (Appendix A.6) ──
    # Normalize so mean pairwise distance ≈ √D
    scale = dist.mean().clamp(min=1e-6)
    dist_normed = dist * (np.sqrt(D) / scale)

    # ── Self-masking, to prevent trivial zero-distance matches between generated samples and themselves ──
    diag_mask = torch.zeros(N, N + M, device=x.device)
    diag_mask[:, :N] = torch.eye(N, device=x.device)
    dist_normed = dist_normed + diag_mask * 100.0

    # ── Force accumulation (NO per-temperature normalization) ──
    V = torch.zeros_like(x)

    for tau in tau_list:
        # Sinkhorn-like affinities: a_ij = exp(-d_ij / tau)
        # with double softmax normalization (over rows and columns) to ensure better gradients and prevent collapse
        # The tau (temperature) controls the sharpness of the affinities: smaller tau → sharper affinities, more emphasis on closer targets
        logits = -dist_normed / tau

        # Double softmax (paper's Alg. 2: softmax over y, then over x)
        aff_row = torch.softmax(logits, dim=-1)
        aff_col = torch.softmax(logits, dim=-2)
        affinity = torch.sqrt((aff_row * aff_col).clamp(min=1e-6))

        aff_neg = affinity[:, :N]                   # negative affinities (to generated samples)
        aff_pos = affinity[:, N:]                   # positive affinities (to data samples)
        sum_pos = aff_pos.sum(-1, keepdim=True)
        sum_neg = aff_neg.sum(-1, keepdim=True)

        # Drifting coefficients: attract by positives, repel by negatives
        r_coeff_neg = -aff_neg * sum_pos
        r_coeff_pos = aff_pos * sum_neg
        R_coeff = torch.cat([r_coeff_neg, r_coeff_pos], dim=1)  # [N, N+M]

        # Force = weighted combination of aim directions (target - x) vectors
        force_R = R_coeff @ targets - R_coeff.sum(-1, keepdim=True) * x
        V = V + force_R  # Raw force, NO normalization

    return V.detach()


def compute_V_ot_direct(x_gen: torch.Tensor, y_pos: torch.Tensor,
                        reg: float = SINKHORN_REG,
                        num_iters: int = SINKHORN_NUM_ITERS,
                        normalize_cost: bool = NORMALIZE_OT_COST):
    """
    Direct OT displacement V(x) = T_Sinkhorn(x) - x.

    Uses the same normalized-cost/log-domain Sinkhorn target as ICNN, so
    differences between the two methods come from ICNN amortization rather
    than from different OT hyperparameters.
    """
    x = x_gen.detach()
    y = y_pos.detach()
    y_target, _, _ = sinkhorn_barycentric_target(
        x, y,
        reg=reg,
        num_iters=num_iters,
        normalize_cost=normalize_cost,
    )
    return (y_target - x).detach()


### 4. Generator (Latent Space Only)

In [8]:
# Latent generator is defined in Section 1 as `LatentGenerator`.
# Quick sanity-check utility:
def inspect_generator(gen: nn.Module, noise_dim: int = NOISE_DIM, batch_size: int = 512, device: str = DEVICE):
    gen = gen.to(device)
    with torch.no_grad():
        z = gen(torch.randn(batch_size, noise_dim, device=device))
    print(f"Generator latent stats -> mean: {z.mean().item():.4f}, std: {z.std().item():.4f}, shape: {tuple(z.shape)}")

### 5. Training Loop for MNIST

In [ ]:
def _next_batch(data_iter, loader):
    try:
        x_real, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        x_real, _ = next(data_iter)
    return x_real, data_iter


def train_mnist(method='kernel', num_iters=NUM_ITERS, batch_size=BATCH_SIZE,
                lr_gen=LR_GEN, inner_steps=INNER_STEPS, device=DEVICE,
                sinkhorn_reg=SINKHORN_REG):
    device = torch.device(device)

    # 1) Data
    transform = transforms.Compose([transforms.ToTensor()])
    dataset = datasets.MNIST(root=MNIST_ROOT, train=True, download=True, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    data_iter = iter(loader)

    # 2) VAE warmup for a stable latent manifold
    vae = MNISTVAE(latent_dim=LATENT_DIM).to(device)
    vae_optim = optim.Adam(vae.parameters(), lr=LR_VAE)
    print(f"[{method}] Warming up VAE for {VAE_WARMUP_STEPS} steps...")
    vae.train()
    for _ in range(VAE_WARMUP_STEPS):
        x_real, data_iter = _next_batch(data_iter, loader)
        x_real = x_real.to(device, non_blocking=True)

        recon, mu, logvar = vae(x_real)
        rec_loss = F.binary_cross_entropy(recon, x_real)
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        vae_loss = rec_loss + 1e-3 * kl_loss

        vae_optim.zero_grad(set_to_none=True)
        vae_loss.backward()
        vae_optim.step()
    print(f"[{method}] VAE warmup done.")

    # 3) Latent generator + optional ICNN drift model
    gen = LatentGenerator(
        noise_dim=NOISE_DIM,
        latent_dim=LATENT_DIM,
        hidden_dims=tuple(GEN_HIDDEN_DIMS),
    ).to(device)
    gen_optim = optim.Adam(gen.parameters(), lr=lr_gen)

    icnn_drift = None
    if method == 'icnn':
        icnn_drift = ICNNDriftField(
            dim=LATENT_DIM,
            hidden_dims=ICNN_HIDDEN_DIMS,
            inner_steps=inner_steps,
            inner_lr=1e-3,
            sinkhorn_reg=sinkhorn_reg,
            sinkhorn_iters=SINKHORN_NUM_ITERS,
            normalize_cost=NORMALIZE_OT_COST,
        ).to(device)
    elif method not in {'kernel', 'ot_direct'}:
        raise ValueError(f"Unknown method: {method}")

    losses, v_norms, snapshots = [], [], []
    vae.eval()
    snapshot_period = max(1, num_iters // 8)

    for it in range(num_iters):
        x_real_img, data_iter = _next_batch(data_iter, loader)
        x_real_img = x_real_img.to(device, non_blocking=True)

        with torch.no_grad():
            mu, _ = vae.encode(x_real_img)
            y_pos = mu  # deterministic latent targets reduce OT noise

        eps = torch.randn(batch_size, NOISE_DIM, device=device)
        x_gen = gen(eps)

        if method == 'kernel':
            V = compute_V_kernel(x_gen, y_pos, tau_list=(0.5, 1.0, 2.0))
        elif method == 'ot_direct':
            V = compute_V_ot_direct(
                x_gen,
                y_pos,
                reg=sinkhorn_reg,
                num_iters=SINKHORN_NUM_ITERS,
                normalize_cost=NORMALIZE_OT_COST,
            )
        elif method == 'icnn':
            V = icnn_drift.compute_V(x_gen, y_pos)

        target_pts = (x_gen.detach() + V).detach()
        loss_drift = ((x_gen - target_pts) ** 2).mean()
        v_norm = (V ** 2).mean().item()

        # Keep generated latents near N(0, I) so they stay decodable.
        second_moment = x_gen.pow(2).mean().clamp_min(1e-6)
        latent_reg = 0.5 * (second_moment - torch.log(second_moment))
        loss = loss_drift + 0.01 * latent_reg

        gen_optim.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        gen_optim.step()

        losses.append(loss.item())
        v_norms.append(v_norm)

        if it % snapshot_period == 0 or it == num_iters - 1:
            with torch.no_grad():
                z_eval = gen(torch.randn(64, NOISE_DIM, device=device))
                x_eval_img = vae.decode(z_eval).cpu()
                snapshots.append((it, x_eval_img.clone()))
            print(
                f"[{method}] Iter {it}/{num_iters} | "
                f"loss={loss.item():.4f} | ||V||^2={v_norm:.5f} | sinkhorn_reg={sinkhorn_reg}"
            )

    return {
        'losses': losses,
        'v_norms': v_norms,
        'snapshots': snapshots,
        'method': method,
        'sinkhorn_reg': sinkhorn_reg,
    }


def run_reg_sweep(method='ot_direct', reg_grid=SINKHORN_REG_GRID,
                  num_iters=500, batch_size=BATCH_SIZE, device=DEVICE):
    """
    Small regularization sweep. Use this before full 3000-step runs.
    With NORMALIZE_OT_COST=True, these reg values are dimensionless.
    """
    sweep_results = []
    for reg in reg_grid:
        print(f"\n{'='*60}\n{method} with sinkhorn_reg={reg}\n{'='*60}")
        sweep_results.append(
            train_mnist(
                method=method,
                num_iters=num_iters,
                batch_size=batch_size,
                lr_gen=LR_GEN,
                inner_steps=INNER_STEPS,
                device=device,
                sinkhorn_reg=reg,
            )
        )
    return sweep_results


In [ ]:
# Find best regularization parameter - using ot_direct bc faster than icnn
reg_sweep_results = run_reg_sweep(
    method="ot_direct",
    reg_grid=SINKHORN_REG_GRID,
    num_iters=500,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

BEST_SINKHORN_REG = reg_sweep_summary[0]["sinkhorn_reg"]

In [ ]:
# Run Training for all methods
results_mnist = []
for m in METHODS:
    print(f"\n{'='*60}\nTraining MNIST: {m}\n{'='*60}")
    results_mnist.append(
        train_mnist(
            method=m,
            num_iters=NUM_ITERS,
            batch_size=BATCH_SIZE,
            lr_gen=LR_GEN,
            inner_steps=INNER_STEPS,
            device=DEVICE,
            sinkhorn_reg=BEST_SINKHORN_REG,
        )
    )


### Visualization

In [11]:
def plot_mnist_results(results_list, save_path=SAVE_PATH):
    n_methods = len(results_list)
    n_snaps = len(results_list[0]['snapshots'])

    fig, axes = plt.subplots(n_methods, n_snaps, figsize=(3.5 * n_snaps, 3.5 * n_methods))
    if n_methods == 1:
        axes = np.array([axes])
    if n_snaps == 1:
        axes = axes.reshape(n_methods, 1)

    for row, res in enumerate(results_list):
        for col, (it, imgs) in enumerate(res['snapshots']):
            ax = axes[row, col]
            grid = make_grid(imgs[:16], nrow=4, normalize=True, value_range=(0, 1))
            grid_np = grid.permute(1, 2, 0).squeeze(-1).numpy()
            ax.imshow(grid_np, cmap='gray')
            ax.set_title(f"{res['method']} | iter {it}")
            ax.axis('off')

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved MNIST results to {save_path}")
    plt.close(fig)

if 'results_mnist' in globals() and len(results_mnist) > 0:
    plot_mnist_results(results_mnist)
else:
    print("Run the training cell first to populate `results_mnist`.")

Run the training cell first to populate `results_mnist`.
